# PhotoMappers State Choropleth Analysis

**Author:** NUS PhotoMapper Team  
**Date:** 2026-09-24

## Objective
Create publication-quality state-level choropleth maps for PhotoMappers observations across the contiguous 48 U.S. states.

## Inputs
- `data/geojson_output/update_dataset_with_disaster_type20260917.geojson`
- `data/geojson_output/us_states_2025.geojson`

## Outputs
- Aggregated geospatial tables in `data/outputs/`
- Choropleth map figures in `data/outputs/`

In [ ]:
# 1) Imports and configuration

from pathlib import Path
from typing import Dict, Tuple

import geopandas as gpd
import mapclassify
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POINTS_FILENAME = "update_dataset_with_disaster_type20260917.geojson"
STATES_FILENAME = "us_states_2025.geojson"

# Exclude AK, HI, PR, DC, and U.S. territories by STATEFP.
EXCLUDED_STATEFP = {"02", "11", "15", "60", "66", "69", "72", "78"}

CRS_GEOGRAPHIC = "EPSG:4326"
CRS_ALBERS_EQUAL_AREA = "EPSG:5070"

FIGSIZE = (12, 7)
DPI = 600
NUM_CLASSES = 15
LEGEND_TITLE = "PhotoMappers observations"


In [ ]:
# 2) Functions

def resolve_input_file(file_name: str) -> Path:
    candidates = [
        PROJECT_DIR / file_name,
        PROJECT_DIR / "geojson_output" / file_name,
        OUTPUT_DIR / file_name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find input file '{file_name}' in expected locations.")


def load_data() -> Tuple[gpd.GeoDataFrame, gpd.GeoDataFrame, Path, Path]:
    points_path = resolve_input_file(POINTS_FILENAME)
    states_path = resolve_input_file(STATES_FILENAME)

    points_gdf = gpd.read_file(points_path)
    states_gdf = gpd.read_file(states_path)
    return points_gdf, states_gdf, points_path, states_path


def inspect_data(points_gdf: gpd.GeoDataFrame, states_gdf: gpd.GeoDataFrame, points_path: Path, states_path: Path) -> Dict[str, object]:
    points_fields = list(points_gdf.columns)
    states_fields = list(states_gdf.columns)

    point_coord_fields = [c for c in ["X", "Y", "longitude", "latitude", "lon", "lat"] if c in points_fields]
    temporal_fields = [c for c in points_fields if any(k in c.lower() for k in ["when", "date", "time", "year"])]
    state_id_fields = [c for c in states_fields if any(k in c.lower() for k in ["state", "name", "stusps", "fp", "geoid", "fips"]) ]

    print("=== INPUT INSPECTION ===")
    print(f"PhotoMappers file: {points_path}")
    print(f"  CRS: {points_gdf.crs}")
    print(f"  Geometry type(s): {points_gdf.geom_type.value_counts().to_dict()}")
    print(f"  Number of features: {len(points_gdf):,}")
    print(f"  Field names: {points_fields}")
    print(f"  Lon/lat attribute candidates: {point_coord_fields}")
    print(f"  Point geometry present: {bool(points_gdf.geom_type.eq('Point').all())}")
    print(f"  Temporal field candidates: {temporal_fields}")
    print()
    print(f"State boundary file: {states_path}")
    print(f"  CRS: {states_gdf.crs}")
    print(f"  Geometry type(s): {states_gdf.geom_type.value_counts().to_dict()}")
    print(f"  Number of features: {len(states_gdf):,}")
    print(f"  Field names: {states_fields}")
    print(f"  State name/FIPS candidates: {state_id_fields}")

    return {
        "points_fields": points_fields,
        "states_fields": states_fields,
        "point_coord_fields": point_coord_fields,
        "temporal_fields": temporal_fields,
        "state_id_fields": state_id_fields,
    }


def prepare_points(points_gdf: gpd.GeoDataFrame, target_crs: str = CRS_GEOGRAPHIC) -> gpd.GeoDataFrame:
    points = points_gdf.copy()

    if points.crs is None:
        points = points.set_crs(target_crs)
    else:
        points = points.to_crs(target_crs)

    if not points.geom_type.eq("Point").all():
        raise ValueError("Point layer contains non-point geometries.")

    return points


def prepare_states(states_gdf: gpd.GeoDataFrame, target_crs: str = CRS_GEOGRAPHIC) -> gpd.GeoDataFrame:
    states = states_gdf.copy()

    required = ["STATEFP", "STUSPS", "NAME"]
    missing = [c for c in required if c not in states.columns]
    if missing:
        raise KeyError(f"Missing required state boundary fields: {missing}")

    states["STATEFP"] = states["STATEFP"].astype(str).str.zfill(2)

    if states.crs is None:
        states = states.set_crs(target_crs)
    else:
        states = states.to_crs(target_crs)

    return states


def filter_contiguous_48(states: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    contiguous = states.loc[~states["STATEFP"].isin(EXCLUDED_STATEFP)].copy()

    # Validation target for this analysis.
    if len(contiguous) != 48:
        raise ValueError(f"Contiguous states count is {len(contiguous)}, expected 48.")

    return contiguous


def assign_year(points: gpd.GeoDataFrame) -> Tuple[gpd.GeoDataFrame, Dict[str, object]]:
    df = points.copy()

    photo_when = pd.to_datetime(df["Photo_When"], errors="coerce", utc=True) if "Photo_When" in df.columns else pd.Series(pd.NaT, index=df.index)
    creation = pd.to_datetime(df["CreationDate"], errors="coerce", utc=True) if "CreationDate" in df.columns else pd.Series(pd.NaT, index=df.index)

    year_source = pd.Series("", index=df.index, dtype="object")
    year_source.loc[photo_when.notna()] = "Photo_When"
    year_source.loc[photo_when.isna() & creation.notna()] = "CreationDate"

    chosen_time = photo_when.fillna(creation)
    df["Year"] = chosen_time.dt.year.astype("Int64")
    df["Year_Source"] = year_source.replace("", pd.NA)

    valid = df["Year"].notna()
    valid_years = df.loc[valid, "Year"].astype(int)

    stats = {
        "year_min": int(valid_years.min()) if len(valid_years) else None,
        "year_max": int(valid_years.max()) if len(valid_years) else None,
        "year_counts": valid_years.value_counts().sort_index(),
        "num_photo_when": int((df["Year_Source"] == "Photo_When").sum()),
        "num_creation_fallback": int((df["Year_Source"] == "CreationDate").sum()),
        "num_missing_year": int((df["Year"].isna()).sum()),
    }

    return df, stats


def spatial_join_points_to_states(points: gpd.GeoDataFrame, states48: gpd.GeoDataFrame) -> Tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    # Keep state attributes needed downstream.
    states_join = states48[["STATEFP", "STUSPS", "NAME", "geometry"]].copy()

    joined = gpd.sjoin(points, states_join, how="left", predicate="within")
    unmatched = joined.loc[joined["STATEFP"].isna()].copy()
    matched = joined.loc[joined["STATEFP"].notna()].copy()

    return matched, unmatched


def aggregate_state_counts(states48: gpd.GeoDataFrame, matched: gpd.GeoDataFrame, year_min: int, year_max: int) -> Tuple[gpd.GeoDataFrame, pd.DataFrame, pd.DataFrame]:
    # Total counts by state.
    total_counts = (
        matched.groupby(["STATEFP", "STUSPS", "NAME"], as_index=False)
        .size()
        .rename(columns={"size": "obs_count"})
    )

    states_total = states48.merge(total_counts, on=["STATEFP", "STUSPS", "NAME"], how="left")
    states_total["obs_count"] = states_total["obs_count"].fillna(0).astype(int)

    # Yearly state counts, including zero-count states.
    matched_yearly = (
        matched.loc[matched["Year"].notna()].copy()
    )
    matched_yearly["Year"] = matched_yearly["Year"].astype(int)

    grouped_yearly = (
        matched_yearly.groupby(["Year", "STATEFP", "STUSPS", "NAME"], as_index=False)
        .size()
        .rename(columns={"size": "obs_count"})
    )

    all_years = pd.DataFrame({"Year": np.arange(year_min, year_max + 1, dtype=int)})
    all_states = states48[["STATEFP", "STUSPS", "NAME"]].drop_duplicates()

    scaffold = all_years.assign(key=1).merge(all_states.assign(key=1), on="key").drop(columns="key")
    yearly_counts = scaffold.merge(grouped_yearly, on=["Year", "STATEFP", "STUSPS", "NAME"], how="left")
    yearly_counts["obs_count"] = yearly_counts["obs_count"].fillna(0).astype(int)

    return states_total, yearly_counts, grouped_yearly


def calculate_class_breaks(states_total: gpd.GeoDataFrame, yearly_counts: pd.DataFrame, num_classes: int = NUM_CLASSES) -> Dict[str, object]:
    # Use one set of breaks across total + yearly maps for comparability.
    combined_counts = np.concatenate([
        states_total["obs_count"].to_numpy(),
        yearly_counts["obs_count"].to_numpy(),
    ])
    positive = combined_counts[combined_counts > 0]

    if len(positive) == 0:
        bins = np.array([1], dtype=float)
        return {"bins": bins, "method": "No positive counts", "use_log": False}

    positive_series = pd.Series(positive)
    median_positive = float(positive_series.median())
    skew_ratio = float(positive_series.max() / median_positive) if median_positive > 0 else np.inf

    use_log = skew_ratio >= 20.0
    k = int(min(num_classes, max(2, positive_series.nunique())))

    if use_log:
        classifier = mapclassify.Quantiles(np.log1p(positive), k=k)
        bins = np.expm1(classifier.bins)
        method = f"Quantiles on log1p(count) with k={k} (skew ratio={skew_ratio:.2f})"
    else:
        classifier = mapclassify.Quantiles(positive, k=k)
        bins = classifier.bins
        method = f"Quantiles on raw count with k={k} (skew ratio={skew_ratio:.2f})"

    bins = np.unique(np.ceil(bins).astype(int))
    bins = bins[bins > 0]

    if len(bins) == 0:
        bins = np.array([1], dtype=int)

    return {
        "bins": bins,
        "method": method,
        "use_log": use_log,
        "skew_ratio": skew_ratio,
    }


def classify_counts_for_plot(counts: pd.Series, bins: np.ndarray) -> pd.Series:
    classes = pd.Series(np.zeros(len(counts), dtype=int), index=counts.index)
    positive_mask = counts > 0
    if positive_mask.any():
        classes.loc[positive_mask] = np.digitize(counts.loc[positive_mask], bins, right=True) + 1
    return classes


def build_palette(num_positive_classes: int = 10):
    # Neutral gray for zero observations
    zero_color = "#E0E0E0"

    # 10 sequential blue classes: light -> dark
    blues = [
        "#E3EEF8",
        "#D0E2F2",
        "#B7D4EA",
        "#94C4DF",
        "#6BAED6",
        "#4A98C9",
        "#2E7EBB",
        "#1764AB",
        "#084A91",
        "#08306B",
    ]

    if num_positive_classes != 10:
        cmap = plt.cm.Blues
        blues = [
            cmap(v)
            for v in np.linspace(0.25, 0.95, num_positive_classes)
        ]

    return [zero_color] + blues


def legend_labels_from_bins(bins: np.ndarray):
    labels = ["0"]
    lower = 1
    for upper in bins:
        labels.append(f"{int(lower)}-{int(upper)}")
        lower = int(upper) + 1
    labels[-1] = f">={int(bins[-2]) + 1}" if len(bins) > 1 else f">={int(bins[0])}"
    return labels


def plot_choropleth(map_df: gpd.GeoDataFrame, bins: np.ndarray, title: str, output_png: Path, extent: Tuple[float, float, float, float]):
    df = map_df.copy()
    df["class_idx"] = classify_counts_for_plot(df["obs_count"], bins)

    palette = build_palette(len(bins))

    fig, ax = plt.subplots(figsize=FIGSIZE)
    fig.patch.set_alpha(0.0)
    ax.set_facecolor("none")

    for class_idx, color in enumerate(palette):
        subset = df.loc[df["class_idx"] == class_idx]
        if len(subset):
            subset.plot(ax=ax, color=color, edgecolor="#6b7280", linewidth=0.35)

    minx, miny, maxx, maxy = extent
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=14, fontweight="semibold", pad=8)

    fig.savefig(output_png, dpi=DPI, transparent=True, bbox_inches="tight")
    plt.close(fig)


def export_results(
    states48: gpd.GeoDataFrame,
    states_total: gpd.GeoDataFrame,
    yearly_counts: pd.DataFrame,
    unmatched_points: gpd.GeoDataFrame,
) -> Dict[str, Path]:
    output_paths = {
        "state_boundary_48": OUTPUT_DIR / "us_contiguous_48_states.geojson",
        "state_total_geojson": OUTPUT_DIR / "photomappers_state_counts.geojson",
        "state_total_csv": OUTPUT_DIR / "state_counts_total.csv",
        "state_yearly_csv": OUTPUT_DIR / "state_counts_yearly.csv",
        "state_yearly_analysis_csv": OUTPUT_DIR / "photomappers_state_year_counts.csv",
        "unmatched_points_geojson": OUTPUT_DIR / "photomappers_unmatched_points.geojson",
    }

    states48.to_file(output_paths["state_boundary_48"], driver="GeoJSON")
    states_total.to_file(output_paths["state_total_geojson"], driver="GeoJSON")

    total_csv = states_total[["STATEFP", "STUSPS", "NAME", "obs_count"]].copy()
    total_csv.to_csv(output_paths["state_total_csv"], index=False)

    yearly_counts.to_csv(output_paths["state_yearly_csv"], index=False)
    yearly_counts.to_csv(output_paths["state_yearly_analysis_csv"], index=False)

    unmatched_points.to_file(output_paths["unmatched_points_geojson"], driver="GeoJSON")

    return output_paths


def validate_outputs(
    points_with_year: gpd.GeoDataFrame,
    states48: gpd.GeoDataFrame,
    matched_points: gpd.GeoDataFrame,
    unmatched_points: gpd.GeoDataFrame,
    states_total: gpd.GeoDataFrame,
    yearly_counts: pd.DataFrame,
    bins: np.ndarray,
    extent: Tuple[float, float, float, float],
    output_paths: Dict[str, Path],
    yearly_png_paths: Dict[int, Path],
    total_png_path: Path,
):
    print("\n=== VALIDATION ===")

    # 1-4: study area checks.
    assert len(states48) == 48, "State count is not 48."
    assert not states48["STATEFP"].isin(EXCLUDED_STATEFP).any(), "Excluded states/territories detected in contiguous layer."

    print("Contiguous state count verified: 48")
    print("Alaska, Hawaii, territories, and DC excluded: True")

    # 5-7: point matching checks.
    valid_points = points_with_year.loc[points_with_year["Year"].notna()].copy()
    print(f"Original valid point count (with Year): {len(valid_points):,}")
    print(f"Successfully matched point count: {len(matched_points):,}")
    print(f"Unmatched point count: {len(unmatched_points):,}")

    # 8: yearly sums match yearly totals of matched points.
    matched_yearly = matched_points.loc[matched_points["Year"].notna()].copy()
    matched_yearly["Year"] = matched_yearly["Year"].astype(int)
    expected_by_year = matched_yearly.groupby("Year").size().sort_index()
    observed_by_year = yearly_counts.groupby("Year")["obs_count"].sum().sort_index()
    pd.testing.assert_series_equal(expected_by_year, observed_by_year, check_names=False)
    print("Yearly state-count sums match yearly matched-point totals: True")

    # 9: total sums.
    assert int(states_total["obs_count"].sum()) == len(matched_points), "Total state counts do not match matched points."
    print("Total state-count sum matches matched points: True")

    # 10-11: same extent and same class breaks are enforced by fixed inputs.
    print(f"Map extent fixed for all maps: {extent}")
    print(f"Class breaks fixed for all maps: {bins.tolist()}")

    # 12-13: file creation and transparent-output checks.
    required_files = list(output_paths.values()) + [total_png_path] + list(yearly_png_paths.values())
    missing_files = [str(p) for p in required_files if not p.exists()]
    assert not missing_files, f"Missing output files: {missing_files}"

    print("All requested output files created: True")
    print(f"Total PNG path: {total_png_path}")
    print(f"Yearly PNG count: {len(yearly_png_paths)}")
    print("Transparent backgrounds requested via fig.savefig(..., transparent=True, dpi=600, bbox_inches='tight').")

In [ ]:
# 3) Run analysis workflow

points_raw, states_raw, points_input_path, states_input_path = load_data()
schema_info = inspect_data(points_raw, states_raw, points_input_path, states_input_path)

points = prepare_points(points_raw, target_crs=CRS_GEOGRAPHIC)
states = prepare_states(states_raw, target_crs=CRS_GEOGRAPHIC)
states48 = filter_contiguous_48(states)

points_with_year, year_stats = assign_year(points)

print("\n=== TEMPORAL SUMMARY ===")
print(f"Year min: {year_stats['year_min']}")
print(f"Year max: {year_stats['year_max']}")
print("Records by year:")
print(year_stats["year_counts"].to_string())
print(f"Records using Photo_When: {year_stats['num_photo_when']:,}")
print(f"Records using CreationDate fallback: {year_stats['num_creation_fallback']:,}")
print(f"Records with missing year: {year_stats['num_missing_year']:,}")

matched_points, unmatched_points = spatial_join_points_to_states(points_with_year, states48)

states_total, yearly_counts, grouped_yearly_nonzero = aggregate_state_counts(
    states48=states48,
    matched=matched_points,
    year_min=year_stats['year_min'],
    year_max=year_stats['year_max'],
)

class_info = calculate_class_breaks(states_total, yearly_counts, num_classes=NUM_CLASSES)
bins = class_info['bins']
print("\n=== CLASSIFICATION DECISION ===")
print(class_info['method'])
print(f"Class breaks used for all maps: {bins.tolist()}")

# Map projection for final choropleths.
states48_map = states48.to_crs(CRS_ALBERS_EQUAL_AREA)
states_total_map = states_total.to_crs(CRS_ALBERS_EQUAL_AREA)
extent = tuple(states48_map.total_bounds)
print(f"Map projection for choropleths: {CRS_ALBERS_EQUAL_AREA}")

# A) Total map
total_png = OUTPUT_DIR / "choropleth_total.png"
plot_choropleth(
    map_df=states_total_map,
    bins=bins,
    title="PhotoMappers Observations, 2017-2026",
    output_png=total_png,
    extent=extent,
)

# B) Yearly maps
yearly_png_paths = {}
for year in range(year_stats['year_min'], year_stats['year_max'] + 1):
    y_df = yearly_counts.loc[yearly_counts['Year'] == year, ['STATEFP', 'STUSPS', 'NAME', 'obs_count']].copy()
    map_df_year = states48.merge(y_df, on=['STATEFP', 'STUSPS', 'NAME'], how='left')
    map_df_year['obs_count'] = map_df_year['obs_count'].fillna(0).astype(int)
    map_df_year_map = map_df_year.to_crs(CRS_ALBERS_EQUAL_AREA)

    year_png = OUTPUT_DIR / f"choropleth_{year}.png"
    plot_choropleth(
        map_df=map_df_year_map,
        bins=bins,
        title=f"{year}",
        output_png=year_png,
        extent=extent,
    )
    yearly_png_paths[year] = year_png

output_paths = export_results(
    states48=states48,
    states_total=states_total,
    yearly_counts=yearly_counts,
    unmatched_points=unmatched_points,
)

validate_outputs(
    points_with_year=points_with_year,
    states48=states48,
    matched_points=matched_points,
    unmatched_points=unmatched_points,
    states_total=states_total,
    yearly_counts=yearly_counts,
    bins=bins,
    extent=extent,
    output_paths=output_paths,
    yearly_png_paths=yearly_png_paths,
    total_png_path=total_png,
)

print("\nWorkflow complete.")

In [ ]:
# 4) Separate legend figure for count breaks

legend_png = OUTPUT_DIR / "choropleth_legend.png"

legend_labels = legend_labels_from_bins(bins)
legend_palette = build_palette(len(bins))
legend_handles = [
    Patch(
        facecolor=legend_palette[i],
        edgecolor="#6b7280",
        linewidth=0.4,
        label=legend_labels[i],
    )
    for i in range(len(legend_labels))
]

fig, ax = plt.subplots(figsize=(4.8, 5.6))
fig.patch.set_alpha(0.0)
ax.set_facecolor("none")
ax.axis("off")

legend = ax.legend(
    handles=legend_handles,
    title=LEGEND_TITLE,
    loc="center",
    frameon=False,
    fontsize=10,
    title_fontsize=11,
    ncol=1,
)
legend.get_title().set_fontweight("semibold")

fig.savefig(legend_png, dpi=DPI, transparent=True, bbox_inches="tight")
plt.close(fig)

print(f"Saved separate legend figure: {legend_png}")
print(f"Legend breaks: {bins.tolist()}")

## Stacked Bar Chart

In [ ]:
# 6) Stacked bar chart: disaster-type composition by year

# Reuse data prepared by the choropleth workflow.
bar_df = points_with_year.copy()

if "Year" not in bar_df.columns:
    raise KeyError("'Year' is required. Run Cell 4 first.")

if "Disaster Type" in bar_df.columns:
    disaster_col = "Disaster Type"
elif "Disaster_Type" in bar_df.columns:
    disaster_col = "Disaster_Type"
else:
    raise KeyError("Neither 'Disaster Type' nor 'Disaster_Type' exists in the dataset.")

bar_df = bar_df.loc[bar_df["Year"].notna()].copy()
bar_df["Year"] = bar_df["Year"].astype(int)
bar_df[disaster_col] = bar_df[disaster_col].fillna("Others/Unknown")

# Year × disaster-type counts and percent composition.
counts = pd.crosstab(bar_df["Year"], bar_df[disaster_col]).sort_index()
percent = counts.div(counts.sum(axis=1), axis=0) * 100

# Explicit category order so color mapping stays stable across reruns.
preferred_order = [
    "Tropical Cyclone",
    "Flood",
    "Extreme Weather",
    "Tornado",
    "Wildfire",
    "Earthquake",
    "Others/Unknown",
]
ordered_cols = [c for c in preferred_order if c in percent.columns] + [
    c for c in percent.columns if c not in preferred_order
]
percent = percent[ordered_cols]

# Base palette aligned with preferred_order.
base_color_map = {
    "Tropical Cyclone": "#3182BD",
    "Flood": "#08306B",
    "Extreme Weather": "#9ECAE1",
    "Tornado": "#99000D",
    "Wildfire": "#CB181D",
    "Earthquake": "#FC9272",
    "Others/Unknown": "#238B45",
}

# If unexpected categories appear, assign extra colors deterministically.
extra_cols = [c for c in percent.columns if c not in base_color_map]
if extra_cols:
    extra_palette = plt.cm.Set2(np.linspace(0, 1, len(extra_cols)))
    for c, rgba in zip(extra_cols, extra_palette):
        base_color_map[c] = rgba

colors = [base_color_map[c] for c in percent.columns]

ax = percent.plot(
    kind="bar",
    stacked=True,
    figsize=(11, 6),
    width=0.78,
    color=colors,
)

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("", fontsize=12)
ax.set_ylim(0, 100)
ax.set_yticks([0, 20, 40, 60, 80, 100])
ax.set_yticklabels(["0%", "20%", "40%", "60%", "80%", "100%"])
ax.set_xticks(range(len(percent.index)))
ax.set_xticklabels(percent.index, rotation=45, ha="right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.25)
ax.set_axisbelow(True)

ax.legend(
    title="Disaster Type",
    bbox_to_anchor=(0.5, 1.18),
    loc="upper center",
    ncol=4,
    frameon=False,
)

plt.tight_layout()
bar_png = OUTPUT_DIR / "disaster_type_by_year.png"
plt.savefig(bar_png, dpi=300, bbox_inches="tight", transparent=True)
plt.show()

print(f"Saved: {bar_png}")

In [ ]:
# ---------------------------------------------------------
# Category order
# ---------------------------------------------------------
lifeline_order = [
    "Safety and Security",
    "Health and Medical",
    "Hazardous Materials",
    "Energy",
    "Communications",
    "Transportation",
    "Food, Hydration, Shelter",
    "Water Systems",
]

# ---------------------------------------------------------
# Fixed colors
# ---------------------------------------------------------
lifeline_colors = {
    "Safety and Security": "#FC9272",
    "Health and Medical": "#CB181D",
    "Hazardous Materials": "#08306B",
    "Energy":  "#99000D",
    "Communications": "#238B45",
    "Transportation":  "#2171B5",
    "Food, Hydration, Shelter": "#6BAED6", 
    "Water Systems": "#74C476",
}

# Reuse data prepared by the choropleth workflow.
lf_df = points_with_year.copy()

if "Year" not in lf_df.columns:
    raise KeyError("'Year' is required. Run Cell 4 first.")

if "Community Lifeline" in lf_df.columns:
    lifeline_col = "Community Lifeline"
elif "Community_Lifelines" in lf_df.columns:
    lifeline_col = "Community_Lifelines"
else:
    raise KeyError("Neither 'Community Lifeline' nor 'Community_Lifelines' exists in the dataset.")

lf_df = lf_df.loc[lf_df["Year"].notna()].copy()
lf_df["Year"] = lf_df["Year"].astype(int)
lf_df[lifeline_col] = lf_df[lifeline_col].fillna("Others/Unknown")

# ---------------------------------------------------------
# Count observations by Year × Lifeline
# ---------------------------------------------------------
counts = (
    lf_df.groupby(["Year", lifeline_col])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=lifeline_order, fill_value=0)
)

# Annual sample size before normalization
annual_n = counts.sum(axis=1)

# ---------------------------------------------------------
# Normalize EACH YEAR independently to exactly 100%
# ---------------------------------------------------------
percent = counts.div(annual_n, axis=0).mul(100)

# Check
yearly_totals = percent.sum(axis=1)
print(yearly_totals)
# Every valid year should return 100.0

# ---------------------------------------------------------
# Plot 100% stacked bars
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 6))

bottom = pd.Series(0.0, index=percent.index)

for lifeline in lifeline_order:
    values = percent[lifeline]

    ax.bar(
        percent.index,
        values,
        bottom=bottom,
        width=0.78,
        color=lifeline_colors[lifeline],
        edgecolor="white",
        linewidth=0.5,
        label=lifeline,
    )

    bottom += values

# ---------------------------------------------------------
# Formatting
# ---------------------------------------------------------
# ax.set_xlabel("Year")
# ax.set_ylabel("Share of observations (%)")
# ax.set_title("Community Lifeline Composition by Year", fontsize=12)
ax.set_ylim(0, 100)
ax.set_yticks([0, 20, 40, 60, 80, 100])
ax.set_yticklabels(["0%", "20%", "40%", "60%", "80%", "100%"])
ax.set_xticks(percent.index)
ax.set_xticklabels(percent.index, rotation=45, ha="right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.25)
ax.set_axisbelow(True)

# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------
ax.legend(
    title="",
    bbox_to_anchor=(0.5, 1.18),
    loc="upper center",
    ncol=4,
    frameon=False,
)

plt.tight_layout()

# ---------------------------------------------------------
# Export
# ---------------------------------------------------------
lifeline_png = OUTPUT_DIR / "community_lifelines_100pct.png"
# lifeline_pdf = OUTPUT_DIR / "community_lifelines_100pct.pdf"

fig.savefig(
    lifeline_png,
    dpi=600,
    bbox_inches="tight",
    transparent=True
)

# fig.savefig(
#     lifeline_pdf,
#     bbox_inches="tight",
#     transparent=True
# )

plt.show()

print(f"Saved: {lifeline_png}")
# print(f"Saved: {lifeline_pdf}")

In [ ]:
# Lifeline breakdown table by year (counts + percentages)

if "counts" not in globals() or "percent" not in globals():
    raise RuntimeError("Run Cell 8 first to create 'counts' and 'percent'.")

counts_tbl = counts.copy().astype(int)
percent_tbl = percent.copy().round(1)

yearly_total = counts_tbl.sum(axis=1).rename("Total")

# Build a side-by-side table with count and percent for each lifeline.
pieces = []
for col in counts_tbl.columns:
    part = pd.DataFrame({
        f"{col} (n)": counts_tbl[col],
        f"{col} (%)": percent_tbl[col],
    })
    pieces.append(part)

lifeline_breakdown_table = pd.concat([yearly_total] + pieces, axis=1)
lifeline_breakdown_table.index.name = "Year"

print("Lifeline breakdown by year:")
display(lifeline_breakdown_table)

out_csv = OUTPUT_DIR / "community_lifelines_breakdown_by_year.csv"
lifeline_breakdown_table.to_csv(out_csv)
print(f"Saved: {out_csv}")